# 30 mm Carbon Steel Air-Assisted Laser Cutting Demo

This notebook runs the full orthogonal experiment data loop: config, pretest plan, L9 plan, simulated quality data, range analysis, main-effect figures, and Agent recommendation.

In [ ]:
from pathlib import Path
import sys
import yaml
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from generate_pretest_plan import generate_pretest_plan
from generate_orthogonal_plan import generate_orthogonal_plan
from simulate_quality_result import simulate_plan
from analyze_orthogonal import analyze_orthogonal
from agent_recommendation import run_recommendation


## 1. Read Experiment Config

In [ ]:
with (PROJECT_ROOT / 'configs' / 'experiment_config.yaml').open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
config


## 2. Generate Pretest Plan

In [ ]:
pretest_plan = generate_pretest_plan()
pretest_plan


## 3. Generate L9 Orthogonal Plan

In [ ]:
orthogonal_plan = generate_orthogonal_plan()
orthogonal_plan


## 4. Simulate Quality Results

In [ ]:
log = simulate_plan(seed=2026)
log[['episode_id', 'power_kw', 'speed_m_min', 'air_pressure_mpa', 'focus_mm', 'failure_case', 'quality_score']]


## 5. View Experiment Log

In [ ]:
pd.read_csv(PROJECT_ROOT / 'data' / 'metadata' / 'experiment_log.csv').head()


## 6. Plot Quality Score Distribution

In [ ]:
figure_dir = PROJECT_ROOT / 'outputs' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
ax = log['quality_score'].plot(kind='bar', figsize=(8, 4), title='Quality Score Distribution')
ax.set_xlabel('Run index')
ax.set_ylabel('quality_score')
plt.tight_layout()
plt.savefig(figure_dir / 'quality_score_distribution.png', dpi=180)
plt.show()


## 7. Range Analysis

In [ ]:
analysis = analyze_orthogonal()
analysis.head(12)


## 8. Main-Effect Figures

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(figure_dir / 'main_effect_quality_score.png')))
display(Image(filename=str(figure_dir / 'main_effect_dross_height.png')))
display(Image(filename=str(figure_dir / 'main_effect_roughness.png')))


## 9. Select Failure Case

In [ ]:
failure_rows = log.loc[log['failure_case'] != 'normal']
selected = failure_rows.iloc[0] if not failure_rows.empty else log.sort_values('quality_score').iloc[0]
selected[['episode_id', 'failure_case', 'quality_score']]


## 10. Agent Recommendation

In [ ]:
recommendation = run_recommendation(selected['episode_id'])
recommendation


## 11. README Command Flow

```bash
pip install -r requirements.txt
python src/generate_pretest_plan.py
python src/generate_orthogonal_plan.py
python src/simulate_quality_result.py --plan data/metadata/orthogonal_plan_L9.csv
python src/analyze_orthogonal.py
python src/agent_recommendation.py --episode_id LC_CS30_AIR_L9_0005
```